# Analysis of finished ```.npz``` files

This Jupyter-Notebook should load the ```.npz```-files which contain the background-residual-field without any coil current, the field with coil current and the simulation results for the field with coil current. *Note* that ```.npz``` files (numpy-zip) need to be within the specified directory. This Code is not supposed to by tidy, but include everything necessary for different visualizations of the result.
_______________
Created 20. May, 2026 by Gregor Bock

(0378 1735; ge27doc)

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi
from scipy.interpolate import CubicSpline

import External_functions as fkt

## Definiere alle Größen

In [ ]:
# Load map with current
start, end = 16, 17                                 # Specify which measurement files shall be loaded (start and end index of the list "File_Basename" in "dic_of_files_and_data")
# Load background map
Background_map = 3                                  # Specify which background file shall be loaded

# All the following data should be chosen equivalently to the Known_coil_Layup calculation!

# If .npz file does not hold geometric track data, specify it here
shift_x = 0                                         # shift of the mapped volume in x-direction
shift_y = 0                                         # shift of the mapped volume in y-direction
shift_z = 0                                         # shift of the mapped volume in z-direction

## Load Data
- Background field (and positions)
- Measured field (and positions)
- Simulated field (and positions)
- Insert zero field values of the QSpin manually!!!

<span style="color:red">The directions of the map and the zero field do not match!!!</span>
- MSR-x = QSpin-x
- MSR-y = QSpin-z
- MSR-z = QSpin-y

```dic_of_files_and_data``` is a dictionary ehich holds the Basename of all measurement files within the Key *File_Basename*. The dictionary also holds the field zero values of the QSpin sensor in the accouding array-entry in the dict-Key *field_zero_vals*. (Example: ```dic_of_files_and_data['File_Basename'][0][i]``` is the i-th Basename of a measurement file. By that it is also part of the filename of the simulation result, ```dic_of_files_and_data['field_zero_vals'][0][i]``` is an array of shape (3) holding the field zero value of the i-th measurment file in the QSpin coordinate system)

```Backgrounds``` has the same structure and is used for the Background-maps.

```folder_exp``` and ```folder_calc``` may need to be changed to find the files in the correct directories on **your** computer!

In [ ]:
# This Data must be created by hand! It holds the names, field zero values and offset values of the measurements
dic_of_files_and_data = {'File_Basename': ['after_degauss_with_current_2026-04-27_16-54-25', 'strom_10mA_03_2026-05-13_14-44-44', 'strom_5mA_04_2026-05-13_15-39-56', 'strom_05mA_05_2026-05-13_17-07-47', 'strom_0001A_02_2026-05-13_12-39-37', 'strom_10mA_mittel_07_2026-05-13_19-24-54', 'strom_5mA_mittel_08_2026-05-13_20-28-47', 'strom_05mA_mittel_09_2026-05-13_21-24-25', 'strom_0001A_mittel_06_2026-05-13_18-22-16', 'Kurz_dis_meas_2026-07-27_16-54-11', 'Lang_dis_meas_2026-07-27_20-36-13', 'MittelKurz_dis_meas_2026-07-27_18-05-37', 'MittelLang_dist_meas_2026-07-27_19-22-37', 'Strom_30_no_degauss_kurz_2026-08-13_15-28-30', 'Strom_30_yes_degauss_kurz_2026-08-13_16-48-56', 'Strom_60_no_degauss_kurz_2026-08-13_17-48-13', 'Strom_60_yes_degauss_kurz_2026-08-13_19-09-43', 'Stom_60_YES_Capacitor_kurz_2026-08-18_14-14-49', 'Strom_60_NO_Capacitor_kurz_2026-08-18_15-56-58'],
                         'field_zero_vals': [[0, 0, 0], [26472, 223, 5857], [-12479, -845, 3758], [-73, -1615, 1918], [1240, -1645, 1926], [-11305, -2671, 11], [-6347, -2203, 867], [-1849, -1749, 1689], [-2312, -1834, 1588], [-3138, -30, 3920], [-3145, -27, 3916], [-3124, 11, 3878], [-3131, -34, 3890], [-2986, -15, 3796], [-3003, 0, 3781], [-3010, -30, 3751], [-3058, -12, 3751], [-3082, 48, 3706], [-3086, 0, 3777]],
                         'Offsets': [[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0], [-61.15, 8.64, 12.9], [-103.79, 63.43, 26.69], [-133.24, 20.95, 29.08], [-122.58, 18.64, 60.15], [-127.75, -21.78, 31.23], [-159.83, 40.68, 43.23]]
                         }
# This Data must be created by hand! It holds the names, field zero values and offset values of the backgrounds
Backgrounds = {'File_Basename': ['after_degauss_no_current_2026-04-27_16-07-03', 'background_01_2026-05-13_10-48-57', 'background_dist_meas_2026-07-27_15-56-01', 'Background_Offsets_2026-08-13_14-23-43'],
               'field_zero_vals': [[-1576, -1794, 1892], [-1541, -1741, 1764], [-2972, 44, 3878], [-2885, 26, 3751]],
               'Offsets': [[0, 0, 0], [0, 0, 0], [0, 0, 0], [-18.16, -10.98, 21.93]]
               }

# Convert field zero values of measurement to MSR (Mapper) coordinate system and SI-units
for i, field_zero_vals in enumerate(dic_of_files_and_data['field_zero_vals']):
    # x-direction is correct -> only switch y and z (positiv sign, if the QSpin is inserted to the mapper with the cable at the upper side)
    B_0_x_QSpin = field_zero_vals[0]
    B_0_y_QSpin = field_zero_vals[1]
    B_0_z_QSpin = field_zero_vals[2]
    dic_of_files_and_data['field_zero_vals'][i] = [B_0_x_QSpin * 10**(-12), -B_0_z_QSpin * 10**(-12), -B_0_y_QSpin * 10**(-12)]

# Convert field zero values of Background to MSR (Mapper) coordinate system and SI-units
for i, field_zero_vals in enumerate(Backgrounds['field_zero_vals']):
    # x-direction is correct -> only switch y and z (positiv sign, if the QSpin is inserted to the mapper with the cable at the upper side)
    B_0_x_QSpin = field_zero_vals[0]
    B_0_y_QSpin = field_zero_vals[1]
    B_0_z_QSpin = field_zero_vals[2]
    Backgrounds['field_zero_vals'][i] = [B_0_x_QSpin * 10**(-12), -B_0_z_QSpin * 10**(-12), -B_0_y_QSpin * 10**(-12)]

# Convert Offset values of measurement to MSR (Mapper) coordinate system and SI-units
for i, Offsets in enumerate(dic_of_files_and_data['Offsets']):
    # x-direction is correct -> only switch y and z (positiv sign, if the QSpin is inserted to the mapper with the cable at the upper side)
    Offset_x = Offsets[0]
    Offset_y = Offsets[1]
    Offset_z = Offsets[2]
    dic_of_files_and_data['Offsets'][i] = [Offset_x * 10**(-12), -Offset_y * 10**(-12), -Offset_z * 10**(-12)]

# Convert Offset values of Background to MSR (Mapper) coordinate system and SI-units
for i, Offsets in enumerate(Backgrounds['Offsets']):
    # x-direction is correct -> only switch y and z (positiv sign, if the QSpin is inserted to the mapper with the cable at the upper side)
    Offset_x = Offsets[0]
    Offset_y = Offsets[1]
    Offset_z = Offsets[2]
    Backgrounds['Offsets'][i] = [Offset_x * 10**(-12), -Offset_y * 10**(-12), -Offset_z * 10**(-12)]

# Give local paths as well as prefix and suffix for file names
folder_exp = "D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\Experiments\\"
folder_calc = "D:\\Studium\\Physik\\Bachelorarbeit\\MSR-field_cancelation\\B_tot_results\\"
prefix_calc = "B_tot_result_"
suffix = "\\map\\points"

In [ ]:
# Loading Background data:
folder_path_background = folder_exp + Backgrounds['File_Basename'][Background_map] + suffix

print(f'\nStarting to load data for background file: {Backgrounds["File_Basename"][Background_map]}')

target_point_coord_exp, B_target_point_background, _ = fkt.load_data_from_folder(folder_path_background, folder_path_background, shift_x, shift_y, shift_z)

# Converting QSpin - Coordinate System to MSR - Coordinate System
B_x = B_target_point_background[:, 0]
B_y = B_target_point_background[:, 1]
B_z = B_target_point_background[:, 2]
B_target_point_background[:, 0] = B_x
B_target_point_background[:, 1] = -B_z
B_target_point_background[:, 2] = -B_y

# Substract Offset from measured Data
for comp in range(3):
    B_target_point_background[:, comp] = B_target_point_background[:, comp] - Backgrounds['Offsets'][Background_map][comp]

In [ ]:
# Loading measurement and calculation data:

# Initialise dictonary and constant values
B_target_point = {'File_Basename': [], 'B_field_exp': [], 'B_field_calc': [], 'Biot_Savart': [], 'target_point_coord_calc': [], 'stream_func': [], 'd_coil': [], 'current': [], 'n_windings': [], 'all_coils': []}
safety_distance = 0
shield_height = 0
shield_width = 0
height_inner_wood = 0
width_inner_wood = 0
coil_plane_dist_to_origin_x = 0
coil_plane_dist_to_origin_y = 0
coil_plane_dist_to_origin_z = 0

# Fill data by iterating over the required maps
for idx, Basename in enumerate(dic_of_files_and_data['File_Basename'][start:end]):
    
    print(f'\nStarting to load data for measurement file: {Basename}')
    B_target_point['File_Basename'].append(Basename)

    folder_path_experiment = folder_exp + Basename + suffix
    folder_path_calc = folder_calc + prefix_calc + Basename + '.npz'
    
    # Load data using the function "load_data_from_folder" from the "Ausgelagerte_Funktionen_Versuchsauswertung.py" file.
    B_target_point['B_field_exp'].append(fkt.load_data_from_folder(folder_path_background, folder_path_experiment, shift_x, shift_y, shift_z)[2])

    # Change coordinate System from QSpin to MSR
    B_x = B_target_point['B_field_exp'][idx][:, 0]
    B_y = B_target_point['B_field_exp'][idx][:, 1]
    B_z = B_target_point['B_field_exp'][idx][:, 2]
    B_target_point['B_field_exp'][idx][:, 0] = B_x
    B_target_point['B_field_exp'][idx][:, 1] = -B_z
    B_target_point['B_field_exp'][idx][:, 2] = -B_y

    # Subtract Offset from measured data 
    for comp in range(3):
        B_target_point['B_field_exp'][idx][:, comp] = B_target_point['B_field_exp'][idx][:, comp] - dic_of_files_and_data['Offsets'][idx+start][comp]
    # Subtract background from measurement data
    B_target_point['B_field_exp'][idx] = B_target_point['B_field_exp'][idx] - B_target_point_background  

    # Load data of calculation
    data_sim = np.load(folder_path_calc)

    # Load constants from data
    if idx == 0 or (safety_distance == data_sim['safety_distance'] and shield_height == data_sim['shield_height'] and shield_width == data_sim['shield_width'] and height_inner_wood == data_sim['height_inner_wood'] and width_inner_wood == data_sim['width_inner_wood'] and coil_plane_dist_to_origin_x == data_sim['coil_plane_dist_to_origin_x'] and coil_plane_dist_to_origin_y == data_sim['coil_plane_dist_to_origin_y'] and coil_plane_dist_to_origin_z == data_sim['coil_plane_dist_to_origin_z']):
        safety_distance = data_sim['safety_distance']
        shield_height = data_sim['shield_height']
        shield_width = data_sim['shield_width']
        height_inner_wood = data_sim['height_inner_wood']
        width_inner_wood = data_sim['width_inner_wood']
        coil_plane_dist_to_origin_x = data_sim['coil_plane_dist_to_origin_x']
        coil_plane_dist_to_origin_y = data_sim['coil_plane_dist_to_origin_y']
        coil_plane_dist_to_origin_z = data_sim['coil_plane_dist_to_origin_z']
        plane_extent = shield_width - safety_distance                                     # Area which gets plotted
        print(f'\nSafety Distance = {safety_distance}\n'
              f'Shield Height = {shield_height}\n'
              f'Shield Width = {shield_width}\n'
              f'Inner Wood Height = {height_inner_wood}\n'
              f'Inner Wood Width = {width_inner_wood}\n'
              f'Distance Coil <-> origin (x) = {coil_plane_dist_to_origin_x}\n'
              f'Distance Coil <-> origin (y) = {coil_plane_dist_to_origin_y}\n'
              f'Distance Coil <-> origin (z) = {coil_plane_dist_to_origin_z}\n')
    else:
        raise ValueError('Constant value(s) of loaded data does not coincide with each other')

    # Load Field values of each calculation
    B_target_point['B_field_calc'].append(data_sim['B_field'])
    B_target_point['Biot_Savart'].append(data_sim['Biot_Savart'])
    B_target_point['target_point_coord_calc'].append(data_sim['points'])
    B_target_point['stream_func'].append(data_sim['stream_function'])
    B_target_point['d_coil'].append(data_sim['d_coil'])
    B_target_point['current'].append(data_sim['current'])
    B_target_point['n_windings'].append(data_sim['n_windings'])
    B_target_point['all_coils'].append(data_sim['all_coils'])

## Some simple plots :)

In [ ]:
# Plot the coil layup, such that it is clear, which experiment setup was used. Additionally, print coil-diameter, currrent and windings, since they can not be determined by the plot alone

for idx, all_coils in enumerate(B_target_point['all_coils']):
    fig_coil_layup = plt.figure()
    ax_coil_layup = fig_coil_layup.add_subplot(111, projection='3d')
    for k in range(all_coils.shape[0]):
        ax_coil_layup.plot(all_coils[k, :, 0], all_coils[k, :, 1], all_coils[k, :, 2], color='blue')
    ax_coil_layup.set_xlabel(r'$x$')
    ax_coil_layup.set_ylabel(r'$y$')
    ax_coil_layup.set_zlabel(r'$z$')
    plt.show()

    print(f'For the file {B_target_point['File_Basename'][idx]}:')
    print(f'  The coil diameter used in the simulation is ' + r'$d_{coil}$ =' + f' {B_target_point['d_coil'][idx]} m')
    print(f'  The coil current used in the simulation is ' + r'$I_{coil}$ =' + f' {B_target_point['current'][idx]} A')
    print(f'  The number of windings of each coil used in the simulation is ' + r'$n_{windings}$ =' + f' {B_target_point['n_windings'][idx]}')

## Plot magnetic fields (norm)

This should give some intuition on how to access and work with the given arrays. Moreover a slight intuition for the field can be obtained (maybe)

In [ ]:
# Plot total field from Background (without coil currents)
fig_B_background = plt.figure()
ax_B_background = fig_B_background.add_subplot(111, projection='3d')
sc_B_background = ax_B_background.scatter(
    target_point_coord_exp[:, 0],
    target_point_coord_exp[:, 1],
    target_point_coord_exp[:, 2],
    c = np.linalg.norm(B_target_point_background, axis=1),
    s = 100,
    cmap = 'viridis',
    alpha = 0.5,
)
fig_B_background.colorbar(sc_B_background, ax=ax_B_background, label=r'$|\mathbf{B}_{\text{background}}|$')
name = Backgrounds['File_Basename'][Background_map]
ax_B_background.set_title(f'Residual B-field without any currents of\n{name}')
ax_B_background.set_xlabel(r'$x$')
ax_B_background.set_ylabel(r'$y$')
ax_B_background.set_zlabel(r'$z$')
ax_B_background.set_xlim([-0.4, 0.4])
ax_B_background.set_ylim([-0.4, 0.4])
ax_B_background.set_zlim([-0.4, 0.4])

B_calc_coarse = []
for idx in range(len(B_target_point['File_Basename'])):
    B_calc = B_target_point['B_field_calc'][idx]
    B_exp = B_target_point['B_field_exp'][idx]
    target_point_coord_calc = B_target_point['target_point_coord_calc'][idx]
    name = B_target_point['File_Basename'][idx]

    # Plot total field from simulation
    fig_B_tot = plt.figure()
    ax_B_tot = fig_B_tot.add_subplot(111, projection='3d')
    sc_B_tot = ax_B_tot.scatter(
        target_point_coord_calc[:, 0],
        target_point_coord_calc[:, 1],
        target_point_coord_calc[:, 2],
        c = np.linalg.norm(B_calc, axis=1),
        s = 100,
        cmap = 'viridis',
        alpha = 0.5,
    )
    fig_B_tot.colorbar(sc_B_tot, ax=ax_B_tot, label=r'$|\mathbf{B}_{\text{total}}|$')
    ax_B_tot.set_title(f'Simulation results B-field of\n{name}')
    ax_B_tot.set_xlabel(r'$x$')
    ax_B_tot.set_ylabel(r'$y$')

    # Plot coarsened field from simulation for better comparison
    num_calc_target_points_fine = int( (len(target_point_coord_calc) + 1) **(1/3) )
    B_tot_coarse = fkt.interpolate_B_on_coarse_grid(num_calc_target_points_fine, target_point_coord_exp, coil_plane_dist_to_origin_x, coil_plane_dist_to_origin_y, coil_plane_dist_to_origin_z, safety_distance, B_calc)
    B_calc_coarse.append(B_tot_coarse)

    fig_B_tot_coarse = plt.figure()
    ax_B_tot_coarse = fig_B_tot_coarse.add_subplot(111, projection='3d')
    sc_B_tot_coarse = ax_B_tot_coarse.scatter(
        target_point_coord_exp[:, 0],
        target_point_coord_exp[:, 1],
        target_point_coord_exp[:, 2],
        c = np.linalg.norm(B_tot_coarse, axis=1),
        s = 100,
        cmap = 'viridis',
        alpha = 0.5,
    )
    fig_B_tot_coarse.colorbar(sc_B_tot_coarse, ax=ax_B_tot_coarse, label=r'$|\mathbf{B}_{\text{total}}|$')
    ax_B_tot_coarse.set_title(f'Simulated results B-field of\n{name}')
    ax_B_tot_coarse.set_xlabel(r'$x$')
    ax_B_tot_coarse.set_ylabel(r'$y$')


    # Plot total field from Experiment (with coil currents)
    fig_B_exp = plt.figure()
    ax_B_exp = fig_B_exp.add_subplot(111, projection='3d')
    sc_B_exp = ax_B_exp.scatter(
        target_point_coord_exp[:, 0],
        target_point_coord_exp[:, 1],
        target_point_coord_exp[:, 2],
        c = np.linalg.norm(B_exp, axis=1),
        s = 100,
        cmap = 'viridis',
        alpha = 0.5,
    )
    fig_B_exp.colorbar(sc_B_exp, ax=ax_B_exp, label=r'$|\mathbf{B}_{\text{experiment}}|$')
    ax_B_exp.set_title(f'B-field from experiment\n{name}')
    ax_B_exp.set_xlabel(r'$x$')
    ax_B_exp.set_ylabel(r'$y$') 

    plt.show()

## Differences of fields and their statistical behaviour

Here the difference of the computed and measured field shall be shown. This should be as small as possible!

<span style="color:red">**Note** that the directions of the field until now do not have to be the same. If a ```+``` or a ```-``` is used must be evaluated manually!!!</span>

In [ ]:
B_diff_arr = []
for idx in range(len(B_target_point['File_Basename'])):
    B_calc = B_target_point['B_field_calc'][idx]
    B_exp = B_target_point['B_field_exp'][idx]

    B_calc_av = np.average(B_calc)
    B_exp_av = np.average(B_exp)
    if B_calc_av - B_exp_av < B_calc_av + B_exp_av:
        B_diff = B_calc_coarse[idx] - B_exp
    else:
        B_diff = B_calc_coarse[idx] + B_exp

    B_diff_arr.append(B_diff)
    B_diff_norm = np.linalg.norm(B_diff, axis=1)

    fig_B_diff = plt.figure()
    ax_B_diff = fig_B_diff.add_subplot(111, projection='3d')
    sc_B_diff = ax_B_diff.scatter(
        target_point_coord_exp[:, 0],
        target_point_coord_exp[:, 1],
        target_point_coord_exp[:, 2],
        c=B_diff_norm,
        s=100,
        cmap='viridis',
        alpha=0.5,
    )
    fig_B_diff.colorbar(sc_B_diff, ax=ax_B_diff, label=r'$|\mathbf{B}_{\text{difference}}|$')
    ax_B_diff.set_title(f"Difference between measured and simulated B-field of\n{B_target_point['File_Basename'][idx]}")
    ax_B_diff.set_xlabel(r'$x$')
    ax_B_diff.set_ylabel(r'$y$')

    # Calculate statistical measures
    B_diff_norm_nT = B_diff_norm * 10**9
    B_diff_variance = np.var(B_diff_norm_nT)
    B_diff_std = np.sqrt(B_diff_variance)
    B_diff_average = np.average(B_diff_norm_nT)

    # Create histogram of differences
    fig_B_diff_hist = plt.figure()
    ax_B_diff_hist = fig_B_diff_hist.add_subplot()

    counts, bins, _ = ax_B_diff_hist.hist(B_diff_norm_nT, bins=32, alpha=0.6, label='Data')

    bin_width = bins[1] - bins[0]
    x = np.linspace(min(B_diff_norm_nT), max(B_diff_norm_nT), 200)

    params = chi.fit(B_diff_norm_nT, floc=0)
    df, loc, scale = params
    chi_pdf = chi.pdf(x, df, loc=loc, scale=scale)
    chi_scaled = chi_pdf * len(B_diff_norm_nT) * bin_width

    ax_B_diff_hist.plot(x, chi_scaled, color='g', label=f'$\\chi$-fit (df={df:.2f})')

    ax_B_diff_hist.set_title(f"Histogram of difference of measurement and simulation\n{B_target_point['File_Basename'][idx]}")
    ax_B_diff_hist.set_xlabel('Difference between simulated and measured B-field in nT')
    ax_B_diff_hist.set_ylabel('Number of points with described difference')
    ax_B_diff_hist.legend()

    plt.show()

    print(f"{B_target_point['File_Basename'][idx]}\n   The average value of the difference in the norm of the B-field is {B_diff_average:.2f} nT\n   The standard deviation of the difference in the norm of the B-field is {B_diff_std:.2f} nT")

## Field-strength along coordinate axis

In [ ]:
def plot_spline(ax, x, y, *, color, ls, label):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) < 2:
        return

    order = np.argsort(x)
    x_sorted = x[order]
    y_sorted = y[order]

    if np.allclose(x_sorted[0], x_sorted[-1]):
        ax.plot(x_sorted, y_sorted, ls=ls, color=color, label=label)
        return

    x_dense = np.linspace(x_sorted[0], x_sorted[-1], 400)
    spline = CubicSpline(x_sorted, y_sorted)
    ax.plot(x_dense, spline(x_dense), ls=ls, color=color, label=label)


def extract_axis_slice(points, field, axis, tol=1e-8):
    points = np.asarray(points, dtype=float)
    field = np.asarray(field, dtype=float)

    if axis == 0:
        mask = np.isclose(points[:, 1], 0.0, atol=tol) & np.isclose(points[:, 2], 0.0, atol=tol)
    elif axis == 1:
        mask = np.isclose(points[:, 0], 0.0, atol=tol) & np.isclose(points[:, 2], 0.0, atol=tol)
    else:
        mask = np.isclose(points[:, 0], 0.0, atol=tol) & np.isclose(points[:, 1], 0.0, atol=tol)

    coords = points[mask, axis]
    vals = field[mask]

    if coords.size == 0:
        return np.empty((0,)), np.empty((0, field.shape[1]))

    order = np.argsort(coords)
    return coords[order], vals[order]


fig_Bx_Xaxis = plt.figure()
ax_Bx_Xaxis = fig_Bx_Xaxis.add_subplot()
ax_Bx_Xaxis.set_title(r'$B_x$ along the $x$-axis')

fig_By_Xaxis = plt.figure()
ax_By_Xaxis = fig_By_Xaxis.add_subplot()
ax_By_Xaxis.set_title(r'$B_y$ along the $x$-axis')

fig_Bz_Xaxis = plt.figure()
ax_Bz_Xaxis = fig_Bz_Xaxis.add_subplot()
ax_Bz_Xaxis.set_title(r'$B_z$ along the $x$-axis')

fig_Bx_Yaxis = plt.figure()
ax_Bx_Yaxis = fig_Bx_Yaxis.add_subplot()
ax_Bx_Yaxis.set_title(r'$B_x$ along the $y$-axis')

fig_By_Yaxis = plt.figure()
ax_By_Yaxis = fig_By_Yaxis.add_subplot()
ax_By_Yaxis.set_title(r'$B_y$ along the $y$-axis')

fig_Bz_Yaxis = plt.figure()
ax_Bz_Yaxis = fig_Bz_Yaxis.add_subplot()
ax_Bz_Yaxis.set_title(r'$B_z$ along the $y$-axis')

fig_Bx_Zaxis = plt.figure()
ax_Bx_Zaxis = fig_Bx_Zaxis.add_subplot()
ax_Bx_Zaxis.set_title(r'$B_x$ along the $z$-axis')

fig_By_Zaxis = plt.figure()
ax_By_Zaxis = fig_By_Zaxis.add_subplot()
ax_By_Zaxis.set_title(r'$B_y$ along the $z$-axis')

fig_Bz_Zaxis = plt.figure()
ax_Bz_Zaxis = fig_Bz_Zaxis.add_subplot()
ax_Bz_Zaxis.set_title(r'$B_z$ along the $z$-axis')

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

for idx, name in enumerate(B_target_point['File_Basename']):
    target_coord_calc = B_target_point['target_point_coord_calc'][idx]
    B_calc = B_target_point['B_field_calc'][idx]
    B_biot_savart = B_target_point['Biot_Savart'][idx]
    B_exp = B_target_point['B_field_exp'][idx]

    coords_calc_x, B_calc_x = fkt.extract_axis_slice(target_coord_calc, B_calc, axis=0)
    coords_calc_y, B_calc_y = fkt.extract_axis_slice(target_coord_calc, B_calc, axis=1)
    coords_calc_z, B_calc_z = fkt.extract_axis_slice(target_coord_calc, B_calc, axis=2)

    coords_biot_x, B_biot_x = fkt.extract_axis_slice(target_coord_calc, B_biot_savart, axis=0)
    coords_biot_y, B_biot_y = fkt.extract_axis_slice(target_coord_calc, B_biot_savart, axis=1)
    coords_biot_z, B_biot_z = fkt.extract_axis_slice(target_coord_calc, B_biot_savart, axis=2)

    coords_exp_x, B_exp_x = fkt.extract_axis_slice(target_point_coord_exp, B_exp, axis=0)
    coords_exp_y, B_exp_y = fkt.extract_axis_slice(target_point_coord_exp, B_exp, axis=1)
    coords_exp_z, B_exp_z = fkt.extract_axis_slice(target_point_coord_exp, B_exp, axis=2)

    color = colors[idx % len(colors)]

    if coords_exp_x.size:
        ax_Bx_Xaxis.plot(coords_exp_x, B_exp_x[:, 0], ls='-', color=color, label=f'Exp.: {name}')
        ax_By_Xaxis.plot(coords_exp_x, B_exp_x[:, 1], ls='-', color=color, label=f'Exp.: {name}') ##
        ax_Bz_Xaxis.plot(coords_exp_x, B_exp_x[:, 2], ls='-', color=color, label=f'Exp.: {name}') ##
    if coords_calc_x.size:
        fkt.plot_spline(ax_Bx_Xaxis, coords_calc_x, B_calc_x[:, 0], color=color, ls='--', label=f'Calc.: {name}')
        fkt.plot_spline(ax_By_Xaxis, coords_calc_x, B_calc_x[:, 1], color=color, ls='--', label=f'Calc.: {name}')
        fkt.plot_spline(ax_Bz_Xaxis, coords_calc_x, B_calc_x[:, 2], color=color, ls='--', label=f'Calc.: {name}')
    if coords_biot_x.size:
        fkt.plot_spline(ax_Bx_Xaxis, coords_biot_x, B_biot_x[:, 0], color=color, ls=':', label=f'Biot-Savart: {name}')
        fkt.plot_spline(ax_By_Xaxis, coords_biot_x, B_biot_x[:, 1], color=color, ls=':', label=f'Biot-Savart: {name}')
        fkt.plot_spline(ax_Bz_Xaxis, coords_biot_x, B_biot_x[:, 2], color=color, ls=':', label=f'Biot-Savart: {name}')

    if coords_exp_y.size:
        ax_Bx_Yaxis.plot(coords_exp_y, B_exp_y[:, 0], ls='-', color=color, label=f'Exp.: {name}')
        ax_By_Yaxis.plot(coords_exp_y, B_exp_y[:, 1], ls='-', color=color, label=f'Exp.: {name}') ##
        ax_Bz_Yaxis.plot(coords_exp_y, B_exp_y[:, 2], ls='-', color=color, label=f'Exp.: {name}') ##
    if coords_calc_y.size:
        fkt.plot_spline(ax_Bx_Yaxis, coords_calc_y, B_calc_y[:, 0], color=color, ls='--', label=f'Calc.: {name}')
        fkt.plot_spline(ax_By_Yaxis, coords_calc_y, B_calc_y[:, 1], color=color, ls='--', label=f'Calc.: {name}')
        fkt.plot_spline(ax_Bz_Yaxis, coords_calc_y, B_calc_y[:, 2], color=color, ls='--', label=f'Calc.: {name}')
    if coords_biot_y.size:
        fkt.plot_spline(ax_Bx_Yaxis, coords_biot_y, B_biot_y[:, 0], color=color, ls=':', label=f'Biot-Savart: {name}')
        fkt.plot_spline(ax_By_Yaxis, coords_biot_y, B_biot_y[:, 1], color=color, ls=':', label=f'Biot-Savart: {name}')
        fkt.plot_spline(ax_Bz_Yaxis, coords_biot_y, B_biot_y[:, 2], color=color, ls=':', label=f'Biot-Savart: {name}')

    if coords_exp_z.size:
        ax_Bx_Zaxis.plot(coords_exp_z, B_exp_z[:, 0], ls='-', color=color, label=f'Exp.: {name}')
        ax_By_Zaxis.plot(coords_exp_z, B_exp_z[:, 1], ls='-', color=color, label=f'Exp.: {name}') ##
        ax_Bz_Zaxis.plot(coords_exp_z, B_exp_z[:, 2], ls='-', color=color, label=f'Exp.: {name}') ##
    if coords_calc_z.size:
        fkt.plot_spline(ax_Bx_Zaxis, coords_calc_z, B_calc_z[:, 0], color=color, ls='--', label=f'Calc.: {name}')
        fkt.plot_spline(ax_By_Zaxis, coords_calc_z, B_calc_z[:, 1], color=color, ls='--', label=f'Calc.: {name}')
        fkt.plot_spline(ax_Bz_Zaxis, coords_calc_z, B_calc_z[:, 2], color=color, ls='--', label=f'Calc.: {name}')
    if coords_biot_z.size:
        fkt.plot_spline(ax_Bx_Zaxis, coords_biot_z, B_biot_z[:, 0], color=color, ls=':', label=f'Biot-Savart: {name}')
        fkt.plot_spline(ax_By_Zaxis, coords_biot_z, B_biot_z[:, 1], color=color, ls=':', label=f'Biot-Savart: {name}')
        fkt.plot_spline(ax_Bz_Zaxis, coords_biot_z, B_biot_z[:, 2], color=color, ls=':', label=f'Biot-Savart: {name}')

axes = [
    (ax_Bx_Xaxis, r'$x$'),
    (ax_By_Xaxis, r'$x$'),
    (ax_Bz_Xaxis, r'$x$'),
    (ax_Bx_Yaxis, r'$y$'),
    (ax_By_Yaxis, r'$y$'),
    (ax_Bz_Yaxis, r'$y$'),
    (ax_Bx_Zaxis, r'$z$'),
    (ax_By_Zaxis, r'$z$'),
    (ax_Bz_Zaxis, r'$z$'),
]

for ax, label in axes:
    ax.grid()
    ax.grid(which='minor', linestyle=':', linewidth=0.5)
    ax.set_xlabel(label)
    if label == '$x$':
        ax.axvline(x=coil_plane_dist_to_origin_x, color='orange', ls='--', label='Coil plane')
        ax.axvline(x=-coil_plane_dist_to_origin_x, color='orange', ls='--')
        ax.axvline(x=shield_width/2, color='red', ls='--', label='Shield plane')
        ax.axvline(x=-shield_width/2, color='red', ls='--')
    elif label == '$y$':
        ax.axvline(x=coil_plane_dist_to_origin_y, color='orange', ls='--', label='Coil plane')
        ax.axvline(x=-coil_plane_dist_to_origin_y, color='orange', ls='--')
        ax.axvline(x=shield_width/2, color='red', ls='--', label='Shield plane')
        ax.axvline(x=-shield_width/2, color='red', ls='--')
    elif label == '$z$':
        ax.axvline(x=coil_plane_dist_to_origin_z, color='orange', ls='--', label='Coil plane')
        ax.axvline(x=-coil_plane_dist_to_origin_z, color='orange', ls='--')
        ax.axvline(x=shield_height/2, color='red', ls='--', label='Shield plane')
        ax.axvline(x=-shield_height/2, color='red', ls='--')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

ax_Bx_Xaxis.set_ylabel(r'$B_x$')
ax_By_Xaxis.set_ylabel(r'$B_y$')
ax_Bz_Xaxis.set_ylabel(r'$B_z$')
ax_Bx_Yaxis.set_ylabel(r'$B_x$')
ax_By_Yaxis.set_ylabel(r'$B_y$')
ax_Bz_Yaxis.set_ylabel(r'$B_z$')
ax_Bx_Zaxis.set_ylabel(r'$B_x$')
ax_By_Zaxis.set_ylabel(r'$B_y$')
ax_Bz_Zaxis.set_ylabel(r'$B_z$')

plt.subplots_adjust(bottom=0.25)
plt.show()


## Field strength off-axis:

In [ ]:
y_line = 0
z_line = 0.2

fig_Bx_line = plt.figure()
ax_Bx_line = fig_Bx_line.add_subplot()
ax_Bx_line.set_title(rf'$B_x$ along $x$ at $y={y_line}\,$m, $z={z_line}$')

fig_By_line = plt.figure()
ax_By_line = fig_By_line.add_subplot()
ax_By_line.set_title(rf'$B_y$ along $x$ at $y={y_line}\,$m, $z={z_line}$')

fig_Bz_line = plt.figure()
ax_Bz_line = fig_Bz_line.add_subplot()
ax_Bz_line.set_title(rf'$B_z$ along $x$ at $y={y_line}\,$m, $z={z_line}$')

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']


for idx in range(len(B_target_point['File_Basename'])):
    name = B_target_point['File_Basename'][idx]
    target_coord_calc = B_target_point['target_point_coord_calc'][idx]
    B_calc = B_target_point['B_field_calc'][idx]
    B_biot_savart = B_target_point['Biot_Savart'][idx]
    B_exp = B_target_point['B_field_exp'][idx]

    x_calc, B_calc_line = fkt.interpolate_line_at_z(target_coord_calc, B_calc, z_line, y_line)
    x_biot, B_biot_line = fkt.interpolate_line_at_z(target_coord_calc, B_biot_savart, z_line, y_line)
    x_exp, B_exp_line = fkt.interpolate_line_at_z(target_point_coord_exp, B_exp, z_line, y_line)

    color = colors[idx % len(colors)]
    if x_exp.size > 0:
        ax_Bx_line.plot(x_exp, B_exp_line[:, 0], ls='-', color=color, label=f'Exp.: {name}')
        ax_By_line.plot(x_exp, B_exp_line[:, 1], ls='-', color=color, label=f'Exp.: {name}')
        ax_Bz_line.plot(x_exp, B_exp_line[:, 2], ls='-', color=color, label=f'Exp.: {name}')
    if x_calc.size > 0:
        fkt.plot_spline(ax_Bx_line, x_calc, B_calc_line[:, 0], color=color, ls='--', label=f'Calc.: {name}')
        fkt.plot_spline(ax_By_line, x_calc, B_calc_line[:, 1], color=color, ls='--', label=f'Calc.: {name}')
        fkt.plot_spline(ax_Bz_line, x_calc, B_calc_line[:, 2], color=color, ls='--', label=f'Calc.: {name}')
    if x_biot.size > 0:
        fkt.plot_spline(ax_Bx_line, x_biot, B_biot_line[:, 0], color=color, ls=':', label=f'Biot-Savart: {name}')
        fkt.plot_spline(ax_By_line, x_biot, B_biot_line[:, 1], color=color, ls=':', label=f'Biot-Savart: {name}')
        fkt.plot_spline(ax_Bz_line, x_biot, B_biot_line[:, 2], color=color, ls=':', label=f'Biot-Savart: {name}')

for ax in (ax_Bx_line, ax_By_line, ax_Bz_line):
    ax.grid()
    ax.grid(which='minor', linestyle=':', linewidth=0.5)
    ax.set_xlabel(r'$x$ [m]')
    ax.axvline(x=coil_plane_dist_to_origin_x, color='orange', ls='--', label='Coil plane')
    ax.axvline(x=-coil_plane_dist_to_origin_x, color='orange', ls='--')
    ax.axvline(x=shield_width/2, color='red', ls='--', label='Shield plane')
    ax.axvline(x=-shield_width/2, color='red', ls='--')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

ax_Bx_line.set_ylabel(r'$B_x$ [T]')
ax_By_line.set_ylabel(r'$B_y$ [T]')
ax_Bz_line.set_ylabel(r'$B_z$ [T]')

plt.subplots_adjust(bottom=0.25)
plt.show()


In [ ]:
# Additional debug for experimental point distribution at z=0
points_z0 = target_point_coord_exp[np.isclose(target_point_coord_exp[:, 2], 0.0, atol=1e-8)]
print('z=0 exp points count:', len(points_z0))
print('unique y values at z=0:', np.unique(np.round(points_z0[:, 1], 6)))
print('x values for z=0:', np.unique(np.round(points_z0[:, 0], 6)))
for x in np.unique(np.round(points_z0[:, 0], 6)):
    yvals = np.unique(np.round(points_z0[np.isclose(points_z0[:, 0], x, atol=1e-6), 1], 6))
    if np.any(yvals < 0.35) and np.any(yvals > 0.35):
        print('x candidate:', x, 'y range:', yvals)
print('nearest y values to 0.35 at z=0:')
for x in np.unique(np.round(points_z0[:, 0], 6))[:10]:
    yvals = np.unique(np.round(points_z0[np.isclose(points_z0[:, 0], x, atol=1e-6), 1], 6))
    print(x, yvals)


In [ ]:
# Choose dataset and fields for plotting (choose index from loaded datasets, not files).
dataset_index = 0                                                       # If only one dataset is loaded, this must be 0.

points_calc = B_target_point['target_point_coord_calc'][dataset_index]
field_calc = B_target_point['B_field_calc'][dataset_index]
field_biot = B_target_point['Biot_Savart'][dataset_index]
name = B_target_point['File_Basename'][dataset_index]


# Plot the central symmetry planes in XY, YZ and XZ
fkt.plot_plane_streamlines('xy', 0.0, r'centered $xy$ plane', points_calc, field_calc, field_biot, plane_extent, shield_width/2, shield_width/2, coil_plane_dist_to_origin_x, name)
fkt.plot_plane_streamlines('yz', 0.0, r'centered $yz$ plane', points_calc, field_calc, field_biot, plane_extent, shield_width/2, shield_height/2, coil_plane_dist_to_origin_x, name)
fkt.plot_plane_streamlines('xz', 0.0, r'centered $xz$ plane', points_calc, field_calc, field_biot, plane_extent, shield_width/2, shield_height/2, coil_plane_dist_to_origin_x, name)

In [ ]:
points_exp = target_point_coord_exp
field_exp = B_target_point['B_field_exp'][dataset_index]

# Plot the measured field streamlines on the central symmetry planes
fkt.plot_plane_streamlines_measured('xy', 0.0, r'centered $xy$ plane', points_exp, field_exp, plane_extent, shield_x=shield_width/2, shield_y=shield_width/2, coil_plane_dist_to_origin_x=coil_plane_dist_to_origin_x)
fkt.plot_plane_streamlines_measured('yz', 0.0, r'centered $yz$ plane', points_exp, field_exp, plane_extent, shield_x=shield_width/2, shield_y=shield_height/2, coil_plane_dist_to_origin_x=coil_plane_dist_to_origin_x)
fkt.plot_plane_streamlines_measured('xz', 0.0, r'centered $xz$ plane', points_exp, field_exp, plane_extent, shield_x=shield_width/2, shield_y=shield_height/2, coil_plane_dist_to_origin_x=coil_plane_dist_to_origin_x)